# Task 5: Inverse Dynamics

## 5.1 Introduction
In this task, you compute joint moments during a gait cycle using **Inverse Dynamics**.

Given segment joint angles (and their time derivatives) together with measured ground reaction forces (GRF), we solve the equations of motion of the rigid body system:

$$\boldsymbol{\tau} = M(q)\,\ddot{q} - F_0(q, \dot{q}, \text{GRF})$$

where:
- $M(q)$ is the **mass matrix** (computed symbolically via Kane's Method)
- $F_0$ is the **forcing vector** (gravity + GRF contributions)
- $\boldsymbol{\tau}$ are the **generalised forces** — joint moments for hinge DOFs, residual forces/moments for the pelvis root DOFs

The first three entries of $\boldsymbol{\tau}$ correspond to the **pelvis DOFs** (`pelvis_tx`, `pelvis_ty`, `pelvis_tilt`). Since the pelvis is the root segment with no parent joint, these entries are *residuals* — they quantify any dynamic inconsistency between the kinematics and the GRF.


## 5.2 Load Kinematics and Ground Reaction Forces

We use **optimised kinematics** from a predictive simulation (biosym OCP), which are dynamically consistent by construction. This gives near-zero pelvis residuals and serves as the clean reference for this task.

Joint velocities $\dot{q}$ and accelerations $\ddot{q}$ are already stored in the data file (from the OCP solution), so no noisy numerical differentiation is needed.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load OCP kinematics (dynamically consistent, smooth)
q_all = pd.read_csv('data/angles_ocp.csv')
time  = q_all['time'].to_numpy()

# Extract coordinates, velocities, and accelerations directly
q_names = ['pelvis_tx', 'pelvis_ty', 'pelvis_tilt', 'q_hip_r', 'q_knee_r', 'q_ankle_r', 'q_hip_l', 'q_knee_l', 'q_ankle_l']
q     = q_all[q_names].to_numpy()
qdot  = q_all[[f'dq_{col}' for col in q_names]].to_numpy()
qddot = q_all[[f'ddq_{col}' for col in q_names]].to_numpy()

# Load the matching OCP ground reaction forces
grf_data = pd.read_csv('data/grf_ocp.csv')

print("Data loaded. Shapes:")
print(f"  q:      {q.shape}    ({q.shape[0]} frames @ 100 Hz, duration={time[-1]:.2f}s)")
print(f"  qdot:   {qdot.shape}")
print(f"  qddot:  {qddot.shape}")
print(f"  GRF:    {grf_data.shape}")
print(f"  DOFs:   {q_names}")


## 5.3 Compute Joint Moments
The `inverse_dynamics` function (in `id/inverse_dynamics.py`) uses the pre-compiled SymPy model to evaluate $M(q)$ and $F_0$ at each frame and returns:
- **`joint_forces`**: dict of residual forces per segment (non-zero only for `'pelvis'`)
- **`joint_moments`**: dict of joint moments per segment (Z-component = sagittal flexion/extension)


In [ ]:
from id.inverse_dynamics import inverse_dynamics

all_forces  = []
all_moments = []

for frame in range(len(q)):
    forces, moments = inverse_dynamics(
        q=q[frame],
        qdot=qdot[frame],
        qddot=qddot[frame],
        grf_row=grf_data.iloc[frame],
    )
    all_forces.append(forces)
    all_moments.append(moments)

print(f"Inverse dynamics computed for all {len(q)} frames.")


## 5.4 Pelvis Residuals
Since the pelvis has no parent joint, $\tau_{\text{pelvis}}$ should be **zero** for a dynamically consistent system.

With optimised kinematics + consistent GRF, these residuals are at the numerical noise floor — confirming the model is correct.

> **Note:** In real motion capture, kinematics (from marker IK) and GRF (from force plates) are measured independently, which always introduces some inconsistency. Tools like OpenSim's Residual Reduction Algorithm (RRA) explicitly minimise these residuals, but don't guarantee that the model is more correct.


In [ ]:
pelvis_forces  = np.array([f['pelvis'] for f in all_forces])
pelvis_moments = np.array([m['pelvis'] for m in all_moments])

body_weight   = 101.2 * 9.81   # N  (model total mass × g)
mean_abs_fx   = np.mean(np.abs(pelvis_forces[:, 0]))
mean_abs_fy   = np.mean(np.abs(pelvis_forces[:, 1]))
mean_abs_mz   = np.mean(np.abs(pelvis_moments[:, 2]))

print("Pelvis residuals (OCP kinematics — should be near zero):")
print(f"  Fx: {mean_abs_fx:.3f} N   ({mean_abs_fx/body_weight*100:.4f}% BW)")
print(f"  Fy: {mean_abs_fy:.3f} N   ({mean_abs_fy/body_weight*100:.4f}% BW)")
print(f"  Mz: {mean_abs_mz:.3f} N·m")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
labels = ['Residual $F_x$ (N)', 'Residual $F_y$ (N)', 'Residual $M_z$ (N·m)']
colors = ['#E91E63', '#9C27B0', '#FF5722']
data_cols = [pelvis_forces[:, 0], pelvis_forces[:, 1], pelvis_moments[:, 2]]

for ax, dat, label, color in zip(axes, data_cols, labels, colors):
    ax.plot(time, dat, color=color, lw=1.5)
    ax.axhline(0, color='k', lw=0.7, ls='--')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)

fig.suptitle('Pelvis Residuals — OCP Kinematics (near zero = consistent)', fontweight='bold')
plt.tight_layout()
plt.show()


## 5.5 Joint Moments over the Gait Cycle
We extract the Z-component (sagittal flexion/extension) for the right hip, knee, and ankle, segment into individual gait cycles, and compute the ensemble average.

**Sign convention (Z-moment):**
- **Hip**: positive = flexion
- **Knee**: positive = extension
- **Ankle**: positive = dorsiflexion (negative = plantarflexion)

You need your segmentation code for this :)


In [ ]:
from utils import segment

hip_moment   = np.array([m['femur_r'][2] for m in all_moments])
knee_moment  = np.array([m['tibia_r'][2] for m in all_moments])
ankle_moment = np.array([m['foot_r'][2]  for m in all_moments])

moments_df = pd.DataFrame({
    'time':         time,
    'hip_moment':   hip_moment,
    'knee_moment':  knee_moment,
    'ankle_moment': ankle_moment,
})

gait_cycles = segment.segment_gait_cycles(grf_data['force_r_y'], data=moments_df, threshold=60)
mean_moments, std_moments = segment.ensemble_average(gait_cycles)

print(f"Segmented {len(gait_cycles)} gait cycle(s)")
print(f"  Hip:   [{mean_moments['hip_moment'].min():.1f}, {mean_moments['hip_moment'].max():.1f}] N·m")
print(f"  Knee:  [{mean_moments['knee_moment'].min():.1f}, {mean_moments['knee_moment'].max():.1f}] N·m")
print(f"  Ankle: [{mean_moments['ankle_moment'].min():.1f}, {mean_moments['ankle_moment'].max():.1f}] N·m")

gait_pct = np.linspace(0, 100, 100)
fig, axes = plt.subplots(3, 1, figsize=(10, 11))

configs = [
    ('hip_moment',   'Hip Moment',   'Flexion/Extension  (+flex)',       '#2196F3'),
    ('knee_moment',  'Knee Moment',  'Extension/Flexion  (+ext)',        '#FF9800'),
    ('ankle_moment', 'Ankle Moment', 'Dorsi-/Plantarflexion  (+dorsi)', '#4CAF50'),
]

for ax, (col, title, subtitle, color) in zip(axes, configs):
    mean = mean_moments[col]
    std  = std_moments[col]
    ax.fill_between(gait_pct, mean - std, mean + std, color=color, alpha=0.15, label='±1 SD')
    ax.plot(gait_pct, mean, color=color, lw=2.5, label='Mean')
    ax.axhline(0, color='gray', ls='--', lw=0.8)
    ax.set_ylabel('Moment (N·m)', fontsize=11)
    ax.set_title(f'{title}  —  {subtitle}', fontsize=11)
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, ls=':', alpha=0.5)

axes[-1].set_xlabel('Gait Cycle (%)', fontsize=11)
fig.suptitle('Right Leg Joint Moments (OCP kinematics)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
